## Create SparkContext and SparkSession

In [1]:
# Reference: lecture material

# Import SparkConf class into program
from pyspark import SparkConf

# local[*]: run Spark in local mode with as many working processors as logical cores on your machine
# If we want Spark to run locally with 'k' worker threads, we can specify as "local[k]".
master = "local[*]"
# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Big Data Projcet"
# Setup configuration parameters for Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Import SparkContext and SparkSession classes
from pyspark import SparkContext # Spark
from pyspark.sql import SparkSession # Spark SQL

# Initialize Spark Session and create a SparkContext Object
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

## Data Loading

In [2]:
# Load the data from the CSV files
df = spark.read.option('inferSchema', True).csv("wrangled_used_cars.csv", header=True)

In [3]:
# Check whether the dataset has been loaded correctly or not 
df.show(5)

+-----------------+------------+---------------+-----------------+------------+----------------+-------------------+--------------+-------------+----------------+---------+-------------+-------+--------------------+----------+-------+-------+----+------+---------+------+-------------+-------+-----------+------------------+--------------------+-----+------------+------------+--------+---------+----------+
|              vin|back_legroom|      body_type|city_fuel_economy|daysonmarket|engine_cylinders|engine_displacement|franchise_make|front_legroom|fuel_tank_volume|fuel_type|has_accidents| height|highway_fuel_economy|horsepower|  price|  width|year|is_cpo|is_oemcpo|is_new|listing_color|mileage|owner_count|             power|              torque|isCab|transmission|wheel_system|  length|wheelbase|model_name|
+-----------------+------------+---------------+-----------------+------------+----------------+-------------------+--------------+-------------+----------------+---------+------------

In [4]:
# Check the shape of the dataframe
print((df.count(), len(df.columns)))

# Check the shcema of the dataframe
df.printSchema()

(704408, 32)
root
 |-- vin: string (nullable = true)
 |-- back_legroom: string (nullable = true)
 |-- body_type: string (nullable = true)
 |-- city_fuel_economy: integer (nullable = true)
 |-- daysonmarket: integer (nullable = true)
 |-- engine_cylinders: string (nullable = true)
 |-- engine_displacement: integer (nullable = true)
 |-- franchise_make: string (nullable = true)
 |-- front_legroom: string (nullable = true)
 |-- fuel_tank_volume: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- has_accidents: boolean (nullable = true)
 |-- height: string (nullable = true)
 |-- highway_fuel_economy: integer (nullable = true)
 |-- horsepower: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- width: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_cpo: boolean (nullable = true)
 |-- is_oemcpo: boolean (nullable = true)
 |-- is_new: boolean (nullable = true)
 |-- listing_color: string (nullable = true)
 |-- mileage: integer (nullable

Notice that the size of the dataframe is really big (around 700,000 rows). Therefore, we need to apply parallel processing when using the dataframe. Thanfully, when creating a dataframe in PySpark, the system automatically partition the data by default. Let's check the number of partition used here.

In [5]:
# Check the initial number of partition
df.rdd.getNumPartitions()

2

Notice, the default number of partition in PySpark is only 2. Therefore, because the size of the data in this case is really big, it is better to add the number of partition. Here, we decided to use the round-robin method to partition to spread the data evenly.

In [6]:
# Repartition the data using the Round-Robin method
df = df.repartition(50)

# Recheck the new number of partition
df.rdd.getNumPartitions()

50

Sadly, due to the size of the data we cannot show how the system divided the data by using the `glom()` function.

## Data Wrangling

In the schema above, notice that the columns back_legroom, front_legroom, fuel_tank_volume, height, wheelbase, width, and length are listed as string, which should be numeric (integer or double). This may happen because in those columns, the value also contains the unit of measurement, which are not numbers. Therefore, before we convert the data type, we also need to remove those units.

In [ ]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import DoubleType

# Remove the non-numbers and convert the data type to double
df = df.withColumn('back_legroom', regexp_replace('back_legroom', '[^\d]', '').cast(DoubleType())) \
        .withColumn('front_legroom', regexp_replace('front_legroom', '[^\d]', '').cast(DoubleType())) \
        .withColumn('fuel_tank_volume', regexp_replace('fuel_tank_volume', '[^\d]', '').cast(DoubleType())) \
        .withColumn('height', regexp_replace('height', '[^\d]', '').cast(DoubleType())) \
        .withColumn('wheelbase', regexp_replace('wheelbase', '[^\d]', '').cast(DoubleType())) \
        .withColumn('width', regexp_replace('width', '[^\d]', '').cast(DoubleType())) \
        .withColumn('length', regexp_replace('length', '[^\d]', '').cast(DoubleType()))

After fixing the datatype of the columns, some NAs appear. This happen due to some rows actually contain invalid values, such as "--" to indicate missing values. Because the value is not a number, thus the converting process will return a NA value. Hence, we also need to remove the NAs.

In [ ]:
# Remove the NAs
df = df.dropna(how = 'any')

In [ ]:
# Re-check the shape of the dataframe
print((df.count(), len(df.columns)))

# Re-check the shcema of the dataframe
df.printSchema()

## Explarotary Data Analysis (EDA)

After the data is cleaned, before using the data, we need explore the data first to see some pattern or the distribution. Firstly, we need to calculate the basic descriptive statistics, such as the mean, standard deviation, minimum, first quartile, median, third quartile, and maximum, of the numerical columns.

In [ ]:
# Calculate the basic descriptive statistics of the numerical columns
df.select('back_legroom', 'city_fuel_economy', 'daysonmarket', 'engine_displacement', 'front_legroom').summary().show()
df.select('fuel_tank_volume', 'height', 'highway_fuel_economy', 'horsepower', 'price').summary().show()
df.select('width', 'year', 'mileage', 'owner_count', 'length', 'wheelbase').summary().show()

Then, for the categorical columns, we count the frequency of each category appearing in the dataset.

In [ ]:
from pyspark.sql.functions import col

# Count the frequency of each category
df.groupBy('body_type').count().sort(col('count').desc()).show()
df.groupBy('engine_cylinders').count().sort(col('count').desc()).show()
df.groupBy('franchise_make').count().sort(col('count').desc()).show()
df.groupBy('fuel_type').count().sort(col('count').desc()).show()
df.groupBy('listing_color').count().sort(col('count').desc()).show()
df.groupBy('power').count().sort(col('count').desc()).show()
df.groupBy('torque').count().sort(col('count').desc()).show()
df.groupBy('transmission').count().sort(col('count').desc()).show()
df.groupBy('wheel_system').count().sort(col('count').desc()).show()

Lastly, for the boolean columns, we can create a pie chart to compare the number of "True" and "False" instances in the dataset.

In [ ]:
import pandas as pd

# Convert the dataframe to pandas dataframe
# so that we can use the pandas functions
df_pd = df.toPandas()

In [ ]:
# Create a pie chart to compare the 'has_accidents' column
ax1 = df_pd.groupby('has_accidents').size().plot(kind='pie', autopct='%.2f')
ax1.set_ylabel('has_accidents')

In [ ]:
# Create a pie chart to compare the 'is_cpo' column
ax2 = df_pd.groupby('is_cpo').size().plot(kind='pie', autopct='%.2f')
ax2.set_ylabel('is_cpo')

In [ ]:
# Create a pie chart to compare the 'is_oemcpo' column
ax3 = df_pd.groupby('is_oemcpo').size().plot(kind='pie', autopct='%.2f')
ax3.set_ylabel('is_oemcpo')

In [ ]:
# Create a pie chart to compare the 'is_new' column
ax4 = df_pd.groupby('is_new').size().plot(kind='pie', autopct='%.2f')
ax4.set_ylabel('is_new')

In [ ]:
# Create a pie chart to compare the 'isCab' column
ax5 = df_pd.groupby('isCab').size().plot(kind='pie', autopct='%.2f')
ax5.set_ylabel('isCab')

Other than doing the basic exploration of the dataset, we also need to do in depth exploration on the "most important" column, which is the 'price' column. This is because the 'price' column will be used as the target variable. Thus, it is good to explore it in detail.

In [ ]:
df_pd.boxplot('price', figsize = (10,5))
df_pd.hist('price', figsize = (10,5))

From the boxplot, it can be seen that the 'price' column has a really wide range of values, thus resulting in a lot of outliers. However, those data points are true outliers as the price of a used car varies a lot, depending on the features and conditions. Therefore, removing those outliers are not recommended.

Next, from the histogram, it can be seen that the distribution is heavily skewed to the right. Interestingly, when comparing the mean and median of the 'price' column in the summary statistic table, the values are not significantly different, which is only by 2,500. The skewness may be caused by the large number of outliers, especially the ones that are greater than upper boundary. The large number of "big numbers" may pull the data distribution into a right skewed one.